# Car Price Prediction — Exploratory Data Analysis

## Overview

This is the third notebook in the project.
We have clean data. Now we explore it.

EDA means Exploratory Data Analysis.
It is the process of understanding your data by looking at it from every angle
before building any machine learning model.

Think of EDA like a doctor examining a patient before prescribing medicine.
You do not jump straight to treatment. You observe, measure, and understand first.

## What This Notebook Covers

| Section | What we explore |
|---------|----------------|
| 1 | Price distribution — what do most cars cost? |
| 2 | Price by car brand — which brands are most expensive? |
| 3 | Price vs year — do newer cars cost more? |
| 4 | Price vs mileage — does more mileage mean lower price? |
| 5 | Price vs engine size — does bigger engine mean higher price? |
| 6 | Price by city — where are the most expensive cars listed? |
| 7 | Price by transmission — Automatic vs Manual |
| 8 | Price by fuel type — Petrol vs Diesel vs Hybrid |

## 1. Importing Libraries

We need four libraries for this notebook.
Pandas handles the data.
Matplotlib and Seaborn draw the charts.
OS builds the file paths.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

All four libraries loaded successfully. Here is what each one does:

pandas — loads and works with the cleaned data table.
Think of it as Excel inside Python.

matplotlib — the engine that draws charts.
Every chart in Python ultimately uses matplotlib under the hood.

seaborn — a layer on top of matplotlib that makes charts look much better
with much less code. Think of matplotlib as a raw engine and seaborn as a
polished car body built on top of it.

os — builds file paths so the code works on any computer.

## 2. Loading the Cleaned Data

We load the file we saved at the end of notebook 02.
This is the clean version — prices are numbers, mileage is numbers,
make column is filled, and bad rows are removed.
We never load the raw file in this notebook.

In [ ]:
data_path = os.path.join('..', 'data', 'processed', '02_cleaned.csv')
df = pd.read_csv(data_path)
print(f"Rows loaded: {len(df)}")
print(f"Columns: {list(df.columns)}")

Good. The clean dataset is loaded and ready to explore.

The columns we see here are the 9 we kept in the cleaning notebook:
make, year, price_pkr, mileage_km, engine_cc,
transmission, fuel_type, city, and seller_type.

Every chart in this notebook will help us understand
what drives the price of a used car in Pakistan.

## 3. Setting the Chart Style

Before drawing any charts, we set a consistent visual style.
This makes every chart look clean and professional.
We also set the default figure size so charts are not too small.

In [ ]:
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["axes.titlesize"] = 14

Let us break down every single line.

`sns.set_style("whitegrid")` — sets the background of every chart to white
with light grey grid lines. This makes the numbers easier to read
because your eye can trace from the bar to the axis easily.
Other options are "darkgrid", "white", "dark", and "ticks".

`plt.rcParams["figure.figsize"] = (12, 5)` — rcParams means "runtime configuration parameters".
This sets the default width and height of every chart in this notebook.
(12, 5) means 12 inches wide and 5 inches tall.
We set this once here so we do not have to repeat it in every chart.

`plt.rcParams["axes.titlesize"] = 14` — sets the default font size of every chart title to 14.
Consistent font sizes make the notebook look professional and polished.

## 4. Price Distribution — What Do Most Cars Cost?

The first question we always ask about the target variable is:
what does its distribution look like?

Distribution means — are most values small, large, or spread evenly?
This is the single most important chart in any prediction project
because it tells us how our target behaves.

In [ ]:
plt.figure()
sns.histplot(df['price_pkr'], bins=50, kde=True)
plt.title('Distribution of Car Prices (PKR)')
plt.xlabel('Price (PKR)')
plt.ylabel('Number of Cars')
plt.show()

Let us break down every single line.

`plt.figure()` — creates a new empty canvas for the chart.
Think of it like taking out a blank piece of paper before drawing.
Without this, matplotlib might draw on top of a previous chart.

`sns.histplot(df['price_pkr'], bins=50, kde=True)` — draws the histogram.
A histogram divides all prices into 50 equal groups called bins
and draws a bar showing how many cars fall into each group.
kde=True adds a smooth curved line on top showing the overall shape.
KDE stands for Kernel Density Estimate — it is a smoothed version of the histogram.

`plt.title(...)` — adds a title at the top of the chart.

`plt.xlabel(...)` — labels the horizontal axis (x-axis).

`plt.ylabel(...)` — labels the vertical axis (y-axis).

`plt.show()` — displays the finished chart on screen.
Always call this at the end, otherwise the chart may not appear.

What the chart tells us:
Most cars are clustered at the lower price range.
A small number of very expensive cars stretch the right tail.
This is called a right-skewed distribution — very common in price data.

## 5. Removing Price Outliers for Cleaner Charts

A small number of extremely expensive cars make it hard to read the charts.
For example, if one car costs 500 lacs, it squishes all other bars to the left.

We will filter to the 99th percentile for visualization purposes only.
This means we remove the top 1% most expensive cars from the charts.
We are NOT removing them from the actual data — only from the visuals.

In [ ]:
p99 = df['price_pkr'].quantile(0.99)
df_viz = df[df['price_pkr'] <= p99].copy()
print(f"Full dataset: {len(df)} rows")
print(f"Viz dataset:  {len(df_viz)} rows")
print(f"Price cap:    PKR {p99:,.0f}")

Let us break down every single line.

`df['price_pkr'].quantile(0.99)` — quantile finds the value below which
a certain percentage of the data falls.
0.99 means the 99th percentile — the price below which 99% of all cars sit.
Any car above this price is in the top 1% — an extreme outlier for visualization.

`df[df['price_pkr'] <= p99].copy()` — creates a new DataFrame called df_viz
that contains only cars priced at or below the 99th percentile cap.
.copy() makes it independent so changes to df_viz do not affect df.

`df_viz` is only used for drawing charts in this notebook.
All analysis and the machine learning model will use the full df.

This is an important distinction — we are not deleting data,
we are just creating a cleaner view for visualization purposes.

## 6. Price by Car Brand

Now we want to know which car brands have the highest median prices.
We use median instead of mean because a few very expensive cars
can pull the mean upward and give a misleading picture.
Median always shows the middle value — more honest and reliable.

In [ ]:
top_makes = df_viz['make'].value_counts().head(10).index
df_makes = df_viz[df_viz['make'].isin(top_makes)]
make_order = df_makes.groupby('make')['price_pkr'].median().sort_values(ascending=False).index

Let us break down every single line.

`df_viz['make'].value_counts().head(10).index` — value_counts() counts how many
listings exist for each brand. head(10) keeps only the top 10 most common brands.
.index extracts just the brand names as a list.
We do this because brands with very few listings give unreliable median prices.

`df_viz['make'].isin(top_makes)` — isin() checks if each row's make
is in our list of top 10 brands. It returns True or False for every row.
df_makes keeps only rows where the make is one of the top 10 brands.

`df_makes.groupby('make')['price_pkr'].median()` — groupby groups all rows
by brand name, then .median() calculates the median price for each brand.

`.sort_values(ascending=False).index` — sorts brands from most expensive
to least expensive and extracts the brand names in that order.
We store this order in make_order so the chart bars appear sorted.

In [ ]:
plt.figure()
sns.boxplot(data=df_makes, x='price_pkr', y='make', order=make_order)
plt.title('Price Distribution by Car Brand (Top 10)')
plt.xlabel('Price (PKR)')
plt.ylabel('Car Brand')
plt.show()

This chart is called a box plot. It shows more information than a simple bar chart.

Here is how to read a box plot:

The vertical line in the middle of each box is the median price.
The left edge of the box is the 25th percentile — 25% of cars cost less than this.
The right edge of the box is the 75th percentile — 75% of cars cost less than this.
The width of the box is called the IQR — Interquartile Range.
The lines extending left and right are called whiskers — they show the full spread.
The dots beyond the whiskers are outliers — unusually priced cars for that brand.

A wide box means prices vary a lot within that brand.
A narrow box means most cars of that brand are priced similarly.

Brands at the top of the chart have the highest median prices.
This tells us that brand alone is a strong predictor of car price.

## 7. Price vs Year — Do Newer Cars Cost More?

We expect newer cars to cost more than older ones.
This relationship between year and price is called correlation.
A scatter plot is the best way to visualize the relationship
between two numeric columns.

In [ ]:
plt.figure()
sns.scatterplot(data=df_viz, x='year', y='price_pkr', alpha=0.3)
plt.title('Car Price vs Year of Manufacture')
plt.xlabel('Year')
plt.ylabel('Price (PKR)')
plt.show()

Let us break down every single line.

`sns.scatterplot(data=df_viz, x='year', y='price_pkr')` — draws one dot
for every single car in the dataset.
The dot's horizontal position shows the year.
The dot's vertical position shows the price.
If dots go up as we move right, it means newer cars cost more.

`alpha=0.3` — alpha controls the transparency of each dot.
0 means fully transparent (invisible). 1 means fully solid.
0.3 means 70% transparent.
We use this because we have thousands of dots — without transparency
they all overlap and you cannot see anything.
Where many dots overlap, the area looks darker.
This is a simple trick to show density in a scatter plot.

What the chart tells us:
As year increases (moving right), price generally increases (moving up).
This confirms that year and price are positively correlated.
Year will be an important feature in our prediction model.

## 8. Price vs Mileage — Does More Mileage Mean Lower Price?

We expect cars with higher mileage to cost less.
A car driven 200,000 km has more wear and tear than one driven 20,000 km.
This is called a negative correlation — as one goes up, the other goes down.

In [ ]:
plt.figure()
sns.scatterplot(data=df_viz, x='mileage_km', y='price_pkr', alpha=0.3)
plt.title('Car Price vs Mileage')
plt.xlabel('Mileage (km)')
plt.ylabel('Price (PKR)')
plt.show()

What the chart tells us:

Cars with lower mileage tend to have higher prices — the dots are higher
on the left side of the chart where mileage is low.

As mileage increases moving right, the price dots tend to drop lower.
This confirms the negative correlation we expected.

However the relationship is not perfectly straight — there is a lot of scatter.
This means mileage alone cannot predict price perfectly.
Other factors like brand, year, and condition also play a big role.

In machine learning we call this noise — the variation that cannot be explained
by a single feature alone. This is why we use multiple features together.

## 9. Price vs Engine Size

Larger engines generally mean more powerful and more expensive cars.
A 660cc kei car and a 3000cc SUV are very different price categories.
Let us see how engine size relates to price in the Pakistani market.

In [ ]:
plt.figure()
sns.scatterplot(data=df_viz, x='engine_cc', y='price_pkr', alpha=0.3)
plt.title('Car Price vs Engine Size')
plt.xlabel('Engine Size (cc)')
plt.ylabel('Price (PKR)')
plt.show()

What the chart tells us:

Cars cluster heavily at the lower engine sizes — 660cc to 1300cc.
This makes sense for Pakistan where small Japanese kei cars and
entry-level sedans dominate the used car market.

As engine size increases, prices generally increase too.
But there is a lot of spread — a 2000cc car can range widely in price
depending on its brand, year, and condition.

The dense cluster at small engine sizes is important for our model.
It means the model will have much more training data for small engines
than for large ones. This is called class imbalance in feature space.

## 10. Price by City

Different cities in Pakistan have different car markets.
Lahore and Karachi are the largest cities with the most listings.
We want to know if location affects price — do city buyers pay more?

In [ ]:
top_cities = df_viz['city'].value_counts().head(8).index
df_cities = df_viz[df_viz['city'].isin(top_cities)]
city_order = df_cities.groupby('city')['price_pkr'].median().sort_values(ascending=False).index

In [ ]:
plt.figure()
sns.boxplot(data=df_cities, x='price_pkr', y='city', order=city_order)
plt.title('Price Distribution by City (Top 8)')
plt.xlabel('Price (PKR)')
plt.ylabel('City')
plt.show()

What the chart tells us:

Different cities have noticeably different median car prices.
This tells us that city is a useful feature for our model.

Cities with higher median prices likely have more demand for premium cars
or a higher concentration of newer imported vehicles.

Cities with lower median prices may have more older locally assembled cars
or simply a different buyer demographic.

The wide boxes in some cities show that prices vary a lot within that city.
A narrow box means most cars in that city are priced similarly.

## 11. Price by Transmission Type

Automatic cars are generally more expensive than Manual cars in Pakistan.
Automatic transmission is seen as a premium feature — more convenient,
more comfortable, and associated with newer imported Japanese vehicles.
Let us confirm this with data.

In [ ]:
plt.figure()
sns.boxplot(data=df_viz, x='transmission', y='price_pkr')
plt.title('Price Distribution by Transmission Type')
plt.xlabel('Transmission')
plt.ylabel('Price (PKR)')
plt.show()

What the chart tells us:

The median price for Automatic cars is higher than for Manual cars.
This confirms our expectation — automatic transmission commands a price premium.

The boxes tell us the spread too.
Automatic cars have a wider price range because they include everything
from small 660cc kei cars to large imported SUVs.
Manual cars tend to cluster at a lower, tighter price range.

For our machine learning model, transmission will be an important feature.
We will convert it to a number in the feature engineering notebook —
Automatic = 1 and Manual = 0.

## 12. Price by Fuel Type

Pakistan's used car market is dominated by Petrol cars.
But Hybrid and Electric cars are becoming more common and command higher prices.
Diesel cars are typically found in trucks, SUVs, and commercial vehicles.
Let us see how fuel type affects price.

In [ ]:
fuel_order = df_viz.groupby('fuel_type')['price_pkr'].median().sort_values(ascending=False).index

plt.figure()
sns.boxplot(data=df_viz, x='fuel_type', y='price_pkr', order=fuel_order)
plt.title('Price Distribution by Fuel Type')
plt.xlabel('Fuel Type')
plt.ylabel('Price (PKR)')
plt.show()

What the chart tells us:

PHEV (Plug-in Hybrid Electric Vehicle) cars have the highest median price.
These are the most advanced and expensive vehicles in the dataset —
combining both electric and petrol engines.

Hybrid cars are second — popular imported Japanese vehicles like
Toyota Aqua, Prius, and Honda Vezel sit in this category.

Electric cars are third — still a premium segment in Pakistan's market.

Petrol cars cover the widest range because they include everything
from a small 660cc kei car to a large 3000cc SUV.

LPG and CNG cars have the lowest median prices — these are typically
older vehicles that have been converted to cheaper fuel types.

Diesel sits at the bottom for median price but has wide spread —
includes both old pickup trucks and expensive diesel SUVs.

Fuel type is clearly a strong predictor of price and will be an
important feature in our machine learning model.

## 13. Correlation Heatmap

A correlation heatmap shows how strongly every numeric column
is related to every other numeric column.

The value ranges from -1 to +1:
- +1 means perfect positive relationship — as one goes up, the other goes up
- -1 means perfect negative relationship — as one goes up, the other goes down
- 0 means no relationship at all

This is one of the most important charts in any ML project because
it tells us which features are most related to our target variable price_pkr.

In [ ]:
numeric_cols = ['price_pkr', 'year', 'mileage_km', 'engine_cc']
corr_matrix = df_viz[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap — Numeric Features vs Price')
plt.show()

Let us break down every single line.

`numeric_cols = ['price_pkr', 'year', 'mileage_km', 'engine_cc']` — we select
only the numeric columns because correlation only works with numbers.
Categorical columns like city or transmission cannot be included here.

`df_viz[numeric_cols].corr()` — .corr() calculates the correlation between
every pair of columns and returns a table of correlation values.
Each cell shows how strongly two columns are related.

`sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)`:
- annot=True — prints the correlation number inside each cell
- fmt='.2f' — shows the number with 2 decimal places
- cmap='coolwarm' — red means strong positive, blue means strong negative
- center=0 — white is at zero, so neutral relationships appear white

How to read the results for price_pkr:
- year vs price_pkr — should be positive (newer = more expensive)
- mileage_km vs price_pkr — should be negative (more km = cheaper)
- engine_cc vs price_pkr — should be positive (bigger engine = more expensive)

The closer the value is to 1 or -1, the stronger the relationship.
Values close to 0 mean the two columns are barely related.

## 14. EDA Summary

Here is a complete summary of everything we discovered in this notebook.
These findings directly guide which features we will use in the ML model.

In [ ]:
print("=== EDA Summary ===")
print(f"Total cars analyzed:     {len(df_viz)}")
print(f"Median car price:        PKR {df_viz['price_pkr'].median():,.0f}")
print(f"Most listed brand:       {df_viz['make'].value_counts().index[0]}")
print(f"Most expensive brand:    {df_viz.groupby('make')['price_pkr'].median().idxmax()}")
print(f"Most listed city:        {df_viz['city'].value_counts().index[0]}")
print(f"Most common fuel type:   {df_viz['fuel_type'].value_counts().index[0]}")
print(f"Most common transmission:{df_viz['transmission'].value_counts().index[0]}")

This summary gives us the key facts about the Pakistani used car market
in a single glance.

Here is what we confirmed through EDA and what it means for our model:

| Finding | What it means for the ML model |
|---------|-------------------------------|
| Year positively correlated with price | year is a strong feature |
| Mileage negatively correlated with price | mileage_km is a strong feature |
| Engine size positively correlated with price | engine_cc is a useful feature |
| Brand has a big impact on price | make must be encoded and included |
| Automatic cars cost more than Manual | transmission is a useful feature |
| PHEV and Hybrid are most expensive | fuel_type is a useful feature |
| City affects median price | city should be included |

All 8 columns in our cleaned dataset are worth keeping for the model.
No column needs to be dropped based on EDA findings.

The next notebook, 04_feature_engineering, will prepare these columns
for machine learning by converting text categories into numbers
and creating new features like car age from the year column.